**Name:** RATUL SIKDER
**Roll:** 2506102
**Course:** MITE 431 - Big Data Analytics

---

# Part B: Unsupervised Learning — Mall Customer Segmentation

Groups shopping-mall customers into natural segments with **K-Means** (k = 5)
using PySpark MLlib. Dataset: `dataset/Mall_Customers.csv` (Kaggle).

## Task 1 — Create a SparkSession and load the CSV

In [2]:
import os

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

master_url = os.environ.get("SPARK_MASTER_URL", "local[*]")

spark = (
    SparkSession.builder
    .appName("MallCustomerSegmentation")
    .master(master_url)
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", master_url)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 11:14:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.2 | master: spark://spark-master:7077


In [3]:
# Rename the Kaggle column names to simple identifiers
df = spark.read.csv("dataset/Mall_Customers.csv", header=True, inferSchema=True)
renames = {
    "Annual Income (k$)": "AnnualIncome",
    "Spending Score (1-100)": "SpendingScore",
    "Genre": "Gender",
}
for old, new in renames.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)
print("Rows:", df.count())

Rows: 200


## Task 2 — Explore: schema and descriptive statistics

In [4]:
df.head()

Row(CustomerID=1, Gender='Male', Age=19, AnnualIncome=15, SpendingScore=39)

In [5]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)



In [6]:
df.describe("Age", "AnnualIncome", "SpendingScore").show()

+-------+-----------------+-----------------+------------------+
|summary|              Age|     AnnualIncome|     SpendingScore|
+-------+-----------------+-----------------+------------------+
|  count|              200|              200|               200|
|   mean|            38.85|            60.56|              50.2|
| stddev|13.96900733155888|26.26472116527124|25.823521668370173|
|    min|               18|               15|                 1|
|    max|               70|              137|                99|
+-------+-----------------+-----------------+------------------+



## Task 3 — Select numerical features

In [7]:
feature_cols = ["Age", "AnnualIncome", "SpendingScore"]

## Task 4 — Create the feature vector

In [8]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

## Task 5 — Standardize the features

In [9]:
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features", withMean=True, withStd=True
)

## Task 6 — K-Means with k = 5

In [10]:
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=5, seed=42)
pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model = pipeline.fit(df)
clustered = model.transform(df)

silhouette = ClusteringEvaluator(
    featuresCol="features", predictionCol="cluster"
).evaluate(clustered)
print(f"Silhouette score (k=5): {silhouette:.4f}")

Silhouette score (k=5): 0.5834


## Task 7 — Cluster assignment for each customer

In [11]:
clustered.select("CustomerID", "Gender", *feature_cols, "cluster").show(
    clustered.count(), truncate=False
)

+----------+------+---+------------+-------------+-------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|cluster|
+----------+------+---+------------+-------------+-------+
|1         |Male  |19 |15          |39           |3      |
|2         |Male  |21 |15          |81           |4      |
|3         |Female|20 |16          |6            |3      |
|4         |Female|23 |16          |77           |4      |
|5         |Female|31 |17          |40           |3      |
|6         |Female|22 |17          |76           |4      |
|7         |Female|35 |18          |6            |3      |
|8         |Female|23 |18          |94           |4      |
|9         |Male  |64 |19          |3            |1      |
|10        |Female|30 |19          |72           |4      |
|11        |Male  |67 |19          |14           |1      |
|12        |Female|35 |19          |99           |4      |
|13        |Female|58 |20          |15           |1      |
|14        |Female|24 |20          |77           |4     

In [12]:
clustered.groupBy("cluster").count().orderBy("cluster").show()

+-------+-----+
|cluster|count|
+-------+-----+
|      0|   40|
|      1|   57|
|      2|   31|
|      3|   49|
|      4|   23|
+-------+-----+



## Task 8 — Cluster centers

In [13]:
# Standardized-space centers, then per-cluster means in original units
kmeans_model = model.stages[-1]
for i, center in enumerate(kmeans_model.clusterCenters()):
    print(f"Cluster {i}: {[round(float(v), 3) for v in center]}")

Cluster 0: [-0.428, 0.972, 1.213]
Cluster 1: [1.195, -0.48, -0.322]
Cluster 2: [0.396, 1.112, -1.228]
Cluster 3: [-0.844, -0.327, -0.355]
Cluster 4: [-0.954, -1.304, 1.098]


In [14]:
clustered.groupBy("cluster").avg(*feature_cols).orderBy("cluster").show()

+-------+-----------------+------------------+------------------+
|cluster|         avg(Age)| avg(AnnualIncome)|avg(SpendingScore)|
+-------+-----------------+------------------+------------------+
|      0|           32.875|              86.1|            81.525|
|      1|55.54385964912281| 47.94736842105263| 41.89473684210526|
|      2|44.38709677419355|  89.7741935483871|18.483870967741936|
|      3|27.06122448979592| 51.97959183673469| 41.04081632653061|
|      4|25.52173913043478|26.304347826086957| 78.56521739130434|
+-------+-----------------+------------------+------------------+



## Task 9 — Interpretation of the customer clusters

| Cluster | Avg Age | Avg Income (k$) | Avg Spending | Customer Group |
|---|---|---|---|---|
| 0 | 32.9 | High (86.1) | High (81.5) | Target customers |
| 1 | 55.5 | Medium (47.9) | Medium (41.9) | Older regular customers |
| 2 | 44.4 | High (89.8) | Low (18.5) | Rich but low spenders |
| 3 | 27.1 | Medium (52.0) | Medium (41.0) | Young regular customers |
| 4 | 25.5 | Low (26.3) | High (78.6) | Young impulsive buyers |

**Discussion.** K-Means clustering has been applied with k = 5 on Age, Annual Income and Spending Score after standardizing the features with StandardScaler. The obtained silhouette score of 0.58 indicates well separated clusters. Cluster 0 is the most valuable segment for the mall, cluster 2 shows high income customers who spend very little, and cluster 4 shows young customers spending heavily despite low income, while clusters 1 and 3 form the regular customer base differing mainly by age. Here k = 5 is more suitable than k = 3, since a smaller k would merge these distinct groups and make the segments less useful for marketing.

In [15]:
spark.stop()